In [255]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import folium

In [273]:
# Load Data sets and merge them
df_edges = pd.read_csv(r"C:\Users\KrishnaDasaNuDasi\Desktop\PROJECTS_GARU\CHALLENGE_GN\edges.csv")
df_nodes = pd.read_csv(r"C:\Users\KrishnaDasaNuDasi\Desktop\PROJECTS_GARU\CHALLENGE_GN\nodes - Copy.csv")

# Convert relevant columns to string type and strip any leading/trailing spaces
df_edges['from'] = df_edges['from'].astype(str).str.strip()
df_nodes['osmid'] = df_nodes['osmid'].astype(str).str.strip()

# Rename columns to 'id' for merging
df_edges = df_edges.rename(columns={'from': 'id'})
df_nodes = df_nodes.rename(columns={'osmid': 'id'})

# Perform many-to-many merge based on 'id'
df = df_edges.merge(df_nodes, how='left', on='id', suffixes=('_edge', '_node'))
df = df.rename(columns={'osmid': 'id_'})
df = df.rename(columns={'id': 'osmid_from'})
# Check the result
#print(df.head())
df.to_csv("merged_data.csv", index=False)


## Data Preprocessing

## Checking Missingness and fixing them

In [274]:
# Check for Missing Value
# function to identify missingness
def identifying_missingness(df):
    missingData = df.isnull().sum()
    missingData_perc = (missingData / len(df))* 100
    return missingData, missingData_perc

# Return a summary of missing data
missingData, missingData_perc = identifying_missingness(df)

missing_summary = pd.DataFrame({
    'Total Missing': missingData,
    'Percentage Missing': missingData / len(df) * 100
    })
print(missing_summary)


                   Total Missing  Percentage Missing
osmid_from                     0            0.000000
to                             0            0.000000
id_                            0            0.000000
street_name                  447           45.334686
highway                        0            0.000000
lanes                        449           45.537525
maxspeed                     874           88.640974
oneway                         0            0.000000
reversed                       0            0.000000
length                         0            0.000000
bridge                       968           98.174442
service                      868           88.032454
access                       974           98.782961
crossing                       0            0.000000
crossing_markings            906           91.886410
surface                        0            0.000000
lit                            0            0.000000
sidewalk                       0            0.

In [275]:
# Function to fill missing street names and lanes
def impute_missing_values(df):
    # Impute missing street_name 
    for i in range(len(df)):
        if pd.isna(df.loc[i, "street_name"]):
            # Fill missing street_name based on the node's street_count
            if df.loc[i, 'street_count'] > 1:  
                # Check nearby nodes (if available) for the street name
                nearby_streets = df[df['osmid_from'] == df.loc[i, 'osmid_from']]['street_name']
                if not nearby_streets.isna().any():
                    df.loc[i, "street_name"] = nearby_streets.iloc[0]  # Fill with the first available name
                else:
                    df.loc[i, "street_name"] = "Unnamed Road"  # Default if no nearby street name            
            else:
                df.loc[i, "street_name"] = df.loc[i, "street_name"] if pd.notna(df.loc[i, "street_name"]) else "Unnamed Road"

    # Impute missing lane count
    for i in range(len(df)):
        if pd.isna(df.loc[i, "lanes"]):
            if df.loc[i, 'type'] == 'traffic_signal':  # If it's a traffic signal
                df.loc[i, "lanes"] = 2  
            elif df.loc[i, 'type'] == 'turning_circle':  # If it's a turning circle
                df.loc[i, "lanes"] = 1  
            elif df.loc[i, 'type'] == 'stop':  # If it's a stop sign
                df.loc[i, "lanes"] = 1  
            elif df.loc[i, 'type'] == 'crossing':  # If it's a pedestrian crossing
                df.loc[i, "lanes"] = 1 
            else:
                df.loc[i, "lanes"] = 1  # Default for blank types or other unknown types

    # Impute maxspeed based on node type or fallback to mean
    for i in range(len(df)):
        if pd.isna(df.loc[i, "maxspeed"]):
            if df.loc[i, 'type'] == 'traffic_signal':
                df.loc[i, "maxspeed"] = 40  # Assume lower speed at traffic signals
            elif df.loc[i, 'type'] == 'turning_circle':
                df.loc[i, "maxspeed"] = 30  # Assume lower speed for turning circles
            elif df.loc[i, 'type'] == 'stop':
                df.loc[i, "maxspeed"] = 30  # Assume lower speed at stop signs
            elif df.loc[i, 'type'] == 'crossing':
                df.loc[i, "maxspeed"] = 20  # Assume lower speed at pedestrian crossings
            else:
                df.loc[i, "maxspeed"] = df['maxspeed'].mean()  # Use mean speed for unknown or blank types

    # Impute categorical fields 
    df['service'] = df['service'].fillna('Other')
    df['access'] = df['access'].fillna('Other')    
    df['bridge'] = df['bridge'].fillna('no') 
    df['crossing_markings'] = df['crossing_markings'].fillna('no marking')

    return df

# Apply the imputation function
df = impute_missing_values(df)
df['type'] = df['type'].fillna('sidewalk')
df = df.drop(columns=['footway_type'])
df = df.rename(columns={'type': 'footway'})

## checking the new dataframe with no missingness

In [276]:
missingData, missingData_perc = identifying_missingness(df)

missing_summary = pd.DataFrame({
    'Total Missing': missingData,
    'Percentage Missing': missingData / len(df) * 100
    })
print(missing_summary)


                   Total Missing  Percentage Missing
osmid_from                     0                 0.0
to                             0                 0.0
id_                            0                 0.0
street_name                    0                 0.0
highway                        0                 0.0
lanes                          0                 0.0
maxspeed                       0                 0.0
oneway                         0                 0.0
reversed                       0                 0.0
length                         0                 0.0
bridge                         0                 0.0
service                        0                 0.0
access                         0                 0.0
crossing                       0                 0.0
crossing_markings              0                 0.0
surface                        0                 0.0
lit                            0                 0.0
sidewalk                       0              

# Feature Engineering

In [277]:
# Looking for busy roads or not
def calculate_busy_score(row):
    score = 0
    if row['lanes'] > 2:  # More lanes = busier
        score += 1
    if row['maxspeed'] > 50:  # Higher speed limit = busier
        score += 1
    if row['bridge'] == "yes":  # Bridges often indicate busy roads
        score += 1
    if row['length'] > 500:  # Longer roads might be busier
        score += 1
    if row['oneway'] == "TRUE":  # One-way streets might be busier
        score += 1
    return score

df['busy_score'] = df.apply(calculate_busy_score, axis=1)

# Categorize roads as busy or not based on the score
df['is_busy'] = df['busy_score'].apply(lambda x: 'Busy' if x > 2 else 'Not Busy')


In [278]:
# looking for pedestrian-friendly or not
def calculate_pedestrian_friendly_score(row):
    score = 0
    if row['sidewalk'] in ['left', 'right','both']:  # Sidewalk present
        score += 2
    elif row['footway'] == 'sidewalk':  # Footway is a sidewalk
        score += 1
    elif row['footway'] == 'crossing':  # Footway is a crossing
        score += 1
    if row['surface'] in ['asphalt', 'paved', 'concrete']: 
        score += 1
    if row['crossing'] == 'marked':  # Crosswalk present
        score += 1
    if row['lit'] == "yes":  # Road is lit (safer at night)
        score += 1
    
    return score


df['pedestrian_friendly_score'] = df.apply(calculate_pedestrian_friendly_score, axis=1)

# Categorize roads as pedestrian-friendly or not based on the score
df['is_pedestrian_friendly'] = df['pedestrian_friendly_score'].apply(lambda x: 'Pedestrian Friendly' if x > 3 else 'Not Pedestrian Friendly')
# Sort streets by pedestrian-friendliness
top_pedestrian_streets = df[['street_name', 'pedestrian_friendly_score']].sort_values(by='pedestrian_friendly_score', ascending=False)
# Check top 10 pedestrian-friendly streets
unique_top_streets = top_pedestrian_streets.drop_duplicates(subset=['street_name']).head(10)
print(unique_top_streets.head(10)) 

             street_name  pedestrian_friendly_score
22   Munsee Street North                          4
467  Ottawa Street North                          3
485  Cayuga Street North                          3
832   Talbot Street East                          3
20    Norton Street West                          3
3       Kerr Street West                          2
957         Unnamed Road                          2
946   Talbot Street West                          2
52           Ouse Street                          2
935      Victoria Street                          2


In [279]:
df.head()

,osmid_from,to,id_,street_name,highway,lanes,maxspeed,oneway,reversed,length,...,lit,sidewalk,latitude,longitude,street_count,footway,busy_score,is_busy,pedestrian_friendly_score,is_pedestrian_friendly
0,416694987,6463584760,67222638,Munsee Street North,secondary,2.0,50.000000,False,False,6.114355,...,yes,right,42.951206,-79.856181,4,sidewalk,0,Not Busy,4,Pedestrian Friendly
1,416694987,6463584763,67222638,Munsee Street North,secondary,2.0,50.000000,False,True,9.588805,...,yes,right,42.951206,-79.856181,4,sidewalk,0,Not Busy,4,Pedestrian Friendly
2,416694987,6463584766,67226665,Kerr Street East,residential,2.0,53.035714,False,True,9.446809,...,no,no,42.951206,-79.856181,4,sidewalk,1,Not Busy,2,Not Pedestrian Friendly
3,416694987,6463584750,67225880,Kerr Street West,residential,2.0,53.035714,False,True,111.908025,...,no,no,42.951206,-79.856181,4,sidewalk,1,Not Busy,2,Not Pedestrian Friendly
4,416695612,4087046866,67222638,Munsee Street North,secondary,2.0,50.000000,False,False,34.661426,...,yes,right,42.953784,-79.857347,4,sidewalk,0,Not Busy,4,Pedestrian Friendly


In [280]:
# Calculate the center of the map using the mean latitude and longitude
map_center = [df['latitude'].mean(), df['longitude'].mean()]
m = folium.Map(location=map_center, zoom_start=14)

# Function to determine color based on pedestrian-friendly score
def get_marker_color(score):
    if score >= 3:
        return 'green'
    elif score >= 2:
        return 'yellow'
    else:
        return 'red'

# Adding markers with color coding based on pedestrian-friendly scores
for idx, row in df.iterrows():
    marker_color = get_marker_color(row['pedestrian_friendly_score'])

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=8,
        color=marker_color,
        fill=True,
        fill_color=marker_color,
        fill_opacity=0.7,
        popup=f"Pedestrian Friendly Score: {row['pedestrian_friendly_score']}"
    ).add_to(m)

# Save the map to an HTML file
m.save("pedestrian_safety_map_final.html")

In [281]:
#Save the final data
df.to_csv("pedestarianSafe_final.csv", index=False)
